# Microsoft Agent Framework — Azure OpenAI (Responses API)

ဤကုဒ်နမူနာတွင်၊ သင်သည် **Microsoft Agent Framework (MAF)** ကို အသုံးပြုကာ **Responses API** ကို အသုံးပြုသည့် **Azure OpenAI** ဖြင့် ကူညီပံ့ပိုးထားသော ရိုးရှင်းသော agent တစ်ခုကို ဖန်တီးပါမည်။

> **ပြောင်းရွှေ့မှုမှတ်ချက်။** ဤနမူနာသည် ယခင်က Semantic Kernel နှင့် GitHub Models ကို အသုံးပြုခဲ့သည်။ ယခု Microsoft Agent Framework သို့ ပြောင်းရွှေ့ပြီး၊ GitHub Models (အသုံးမပြုတော့၊ ဇူလိုင် ၂၀၂၆ တွင် ပိတ်သိမ်းမည်) ကို Azure OpenAI ဖြင့် အစားထိုးခဲ့ပြီး၊ Responses API ကို ထောက်ပံ့သည်။ MAF တွင်ရှိသည့် `OpenAIChatClient` သည် Azure OpenAI ၏ အတည်ပြုထားသော `/openai/v1/` endpoint ကို ရည်ရွယ်ပြီး Responses API ကို အပေါ်လမ်းကြောင်းအဖြစ် အသုံးပြုသည်။

ဤနမူနာ၏ ရည်ရွယ်ချက်မှာ နောက်ထပ် ကုဒ်နမူနာများတွင် agentic ပုံစံများကို အကောင်အထည်ဖော်ရာ၌ လုပ်ဆောင်ရန်အဆင့်များကို ပြသရန် ဖြစ်သည်။


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## လိုအပ်သည့် Python အထုပ်များကို တင်သွင်းပါ


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ကိရိယာ တစ်ခု သတ်မှတ်ခြင်း

Microsoft Agent Framework တွင်၊ **ကိရိယာ** သည် `@tool` ဖြင့် ဖော်ပြထားသော ရိုးရှင်းသော Python ဖန်ရှင်တစ်ခုဖြစ်ပြီး agent သည် ယင်းကို ခေါ်နိုင်ပါသည်။ အောက်တွင် မတူညီသော အားလပ်ရက် ပို့ဆောင်ရာနေရာကို ပြန်ပေးသော ကိရိယာတစ်ခုကို သတ်မှတ်ထားသည်။


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Agent ကို ဖန်တီးခြင်း

ဒီမှာတော့ `TravelAgent` ဟု အမည်ရသော Agent ကို ဖန်တီးပါသည်။

ဤဥပမာတွင် ကျွန်ုပ်တို့သည် အလွန်ရိုးရှင်းသော ညွှန်ကြားချက်များကို အသုံးပြုသည်။ Agent ၏ အပြုအမူ ပြောင်းလဲမှုကို ကြည့်ရှုရန် ဤညွှန်ကြားချက်များကို မည်သို့မဆို ပြင်ဆင်နိုင်ပါသည်။


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ဂျင်ယက်ကို ပြေးဆွဲခြင်း

ယခု ဂျင်ယက်ကို ပြေးဆွဲနိုင်ပါပြီ။ ဂျင်ယက်က တစ်ခြားဖိုင်များအကြား စကားပြောမှတ်ဥာဏ်များကို မှတ်မိနိုင်ရန် `AgentSession` တစ်ခု ဖန်တီးပြီးတော့၊ user_inputs နှစ်ခု ပေးပို့သည်။ ပထမတွင် ခရီးစဉ်တစ်ခု မေးမြန်းပြီး၊ ဒုတိယတွင် အသုံးပြုသူသည် အကြံပြုချက်မကြိုက်ဘူးလို့ ပြောပြီး အခြားတစ်ခုကို တောင်းဆိုသည် — ဂျင်ယက်မှာ session မှတ်တမ်းနှင့် `get_random_destination` ကိရိယာကို အသုံးပြုပြီး တုံ့ပြန်သည်။

သင်သည် ဤမက်ဆေ့ဂျ်များကို ပြင်ဆင်၍ ဂျင်ယက်၏ တုံ့ပြန်မှုကွဲပြားမှုကို တွေ့နိုင်ပါသည်။ တုံ့ပြန်ချက်များကို တစ်လုံးချင်းစီ စီးဆင်း ကာ ပြသသည်။


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
